# Example of Tracking-60k

In [1]:
import os
import sys
sys.path.append('../src')

import torch
import torch.utils.benchmark as benchmark
from pathlib import Path

from transformer import Transformer
from trainer import run_one_epoch, init_metrics
from utils import get_loss
from utils.get_data import get_data_loader, get_dataset

torch.set_num_threads(10)

In [2]:
device = 'cuda:7'
dataset_name = 'tracking-60k'
batch_size = 1
model_configs = {'block_size': 100, 'n_hashes': 3, 'num_regions': 150, 'num_heads': 8, 'h_dim': 24, 'n_layers': 4, 'num_w_per_dist': 10}
torch.cuda.set_device(device)

In [3]:
dataset_dir = Path('../data/') / dataset_name.split("-")[0]
dataset = get_dataset(dataset_name, dataset_dir)

In [4]:
loaders = get_data_loader(dataset, dataset.idx_split, batch_size=batch_size)

In [5]:
model = Transformer(in_dim=dataset.x_dim, coords_dim=dataset.coords_dim, num_classes=dataset.num_classes, **model_configs).to(device)

In [6]:
checkpoint = torch.load("./ckpt/tracking-60k-model.pt", map_location="cpu")
model.load_state_dict(checkpoint, strict=True)
model = model.to(device)

criterion = get_loss('infonce', {'dist_metric': 'l2_rbf', 'tau': 0.05})
metrics = init_metrics(dataset_name)

In [7]:
with torch.no_grad():
    model.eval()
    test_res = run_one_epoch(model, None, criterion, loaders["test"], "test", 0, device, metrics, None)

print(f"Test accuracy@0.9: {test_res['accuracy@0.9']:.4f}")

[Epoch 0] test , loss: 0.5744, acc: 0.9208, prec: 0.3808, recall: 0.9758: 100%|██████████| 5/5 [00:04<00:00,  1.05it/s]

Test accuracy@0.9: 0.9208


In [4]:
a, b = torch.unique(dataset[0].particle_id, return_counts=True)

In [4]:
for i in range(len(dataset)):
    data = dataset[i]
    a, b = torch.unique(data.particle_id[data.pt > 0.9], return_counts=True)

    print(len(a), b.max().item(), (data.pt > 0.9).sum().item(), data.pt.shape[0])


1792 16 11837 65401
1417 16 9374 57397
1148 16 7242 45870
1343 17 8878 54295
1662 16 10856 65572
1610 17 10648 60354
1314 16 8305 52574
1429 16 9255 55955
1321 16 8861 53467
1582 16 10265 59983
1125 16 7330 48272
1561 16 10328 60938
1771 16 11588 71848
1295 16 8418 56699
1233 16 7938 50341
1360 17 8969 57589
1273 16 8096 50820
1620 17 10611 63415
1237 17 8120 52203
1760 16 11285 64885
1342 16 8660 54489
1198 16 8047 49521
1618 16 10564 63331
1558 16 9918 59611
1773 16 11479 68556
1247 16 8044 51964
1626 16 10902 60611
1556 16 10195 65235
955 16 6308 44198
1637 16 10483 63116
1615 16 10477 62310
988 16 6610 42864
1338 17 9005 55545
1770 16 11321 66665
1368 16 9017 55119
1187 16 7746 51254
1476 17 9592 57421
1430 17 9115 57530
1195 16 7764 46991
1316 16 8715 54491
1236 16 8216 52944
1284 16 8447 55453
1197 16 7699 54968
1651 16 11113 62391
1505 17 10206 61205
1651 16 10789 64110
1568 16 9956 56338
959 16 6271 41758
1313 16 8726 54651
1374 17 9084 54315


In [6]:
a

tensor([  4503737066323968,   4503943224754176,   4504424261091328,
         ..., 959268507336310784, 959268576055787520,
        959268644775264256])

In [5]:
len(dataset)

50

In [14]:
a.shape

torch.Size([1792])

In [21]:
b.float().max()

tensor(16.)

In [5]:
(dataset[i].pt > 0.9).sum(), dataset[i].pt.shape

NameError: name 'i' is not defined

# Benchmark Inference Speed

In [8]:
# cuda 12.1
model = torch.compile(model)

In [9]:
torch.set_float32_matmul_precision('high')
for data in loaders["test"]:
    if data.x.shape[0] > 60000:
        data = data.to(device)
        break

model.eval()
with torch.no_grad():
    t1 = benchmark.Timer(
        stmt=f"model(data.x, data.coords, data.batch)", setup=f"from __main__ import model, data"
    )
    m = t1.blocked_autorange(min_run_time=5)
print(m)

model(data.x, data.coords, data.batch)
setup: from __main__ import model, data
  Median: 28.17 ms
  IQR:    0.07 ms (28.13 to 28.20)
  178 measurements, 1 runs per measurement, 1 thread
